In [ ]:
import duckdb
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import date
from sentence_transformers import SentenceTransformer

In [ ]:
con = duckdb.connect("../../ProjectData.duckdb")
con.execute("CREATE SCHEMA IF NOT EXISTS gold")

silver_files = sorted(Path("../../data/silver").glob("cleaned_data_load_date=*.parquet"))
if not silver_files:
    raise FileNotFoundError("No Silver Parquet found. Run clean_transform_to_silver.ipynb first.")

silver_path = silver_files[-1]
print(f"Loading Silver from: {silver_path}")

silver_df = pd.read_parquet(silver_path)
print(f"Rows loaded:  {len(silver_df):,}")
print(f"Columns:      {list(silver_df.columns)}")

In [ ]:
MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"
# alternative: sentence-transformers/LaBSE

print(f"Loading model: {MODEL_NAME} ...")
model = SentenceTransformer(MODEL_NAME)

print(f"Embedding dim:   {model.get_sentence_embedding_dimension()}")
print(f"Max seq length:  {model.max_seq_length} tokens")

In [ ]:
def encode_field(model, series, batch_size=64, field_name=""):
    """
    Encode a pandas Series of text into sentence embeddings.
    Returns:
        embeddings : np.ndarray of shape (n, dim), unnormalised
        valid_mask : np.ndarray of bool — False where original text was None/empty
    """
    texts = series.fillna("").str.strip()
    valid_mask = (texts != "").values
    inputs = texts.where(texts != "", other=" ").tolist()

    print(f"Encoding {field_name} ({valid_mask.sum():,} / {len(texts):,} non-empty) ...")
    embeddings = model.encode(
        inputs,
        batch_size=batch_size,
        show_progress_bar=True,
        normalize_embeddings=False,  
    )
    return embeddings.astype(np.float32), valid_mask


body_emb,     body_valid     = encode_field(model, silver_df["review_body"],     field_name="review_body")
headline_emb, headline_valid = encode_field(model, silver_df["review_headline"], field_name="review_headline")
title_emb,    title_valid    = encode_field(model, silver_df["product_title"],   field_name="product_title")

print(f"\nShapes — body: {body_emb.shape}, headline: {headline_emb.shape}, title: {title_emb.shape}")

In [ ]:
def cosine_sim_rowwise(a, b, mask_a, mask_b):
    """
    Row-wise cosine similarity between two unnormalised embedding matrices.
    Returns NaN where either input was originally missing/empty.
    """
    a_norm = a / np.clip(np.linalg.norm(a, axis=1, keepdims=True), 1e-8, None)
    b_norm = b / np.clip(np.linalg.norm(b, axis=1, keepdims=True), 1e-8, None)
    sim = np.einsum("ij,ij->i", a_norm, b_norm).astype(float)
    sim[~(mask_a & mask_b)] = np.nan
    return sim

silver_df["headline_body_cosine_sim"] = cosine_sim_rowwise(
    headline_emb, body_emb, headline_valid, body_valid
)

silver_df["title_body_cosine_sim"] = cosine_sim_rowwise(
    title_emb, body_emb, title_valid, body_valid
)

body_norms = np.linalg.norm(body_emb, axis=1).astype(float)
body_norms[~body_valid] = np.nan
silver_df["body_embedding_norm"] = body_norms

print("Similarity feature stats:")
silver_df[["headline_body_cosine_sim", "title_body_cosine_sim", "body_embedding_norm"]].describe().round(3)

In [ ]:
# Build a DataFrame with only the join key + new features.
features_df = silver_df[[
    "_index",
    "product_id",
    "label",
    "headline_body_cosine_sim",
    "title_body_cosine_sim",
    "body_embedding_norm",
]].copy()

con.register("embedding_features_temp", features_df)

con.execute("DROP TABLE IF EXISTS gold.review_embedding_features")
con.execute("""
    CREATE TABLE gold.review_embedding_features AS
    SELECT * FROM embedding_features_temp
""")

print("gold.review_embedding_features created.")
print("Rows:", con.execute("SELECT COUNT(*) FROM gold.review_embedding_features").fetchone()[0])

In [ ]:
# cosine similarities should be in [-1, 1]
# null counts should match the number of missing headlines / titles in Silver
# body_embedding_norm should have no nulls
con.execute("""
    SELECT
        COUNT(*)                                                              AS total_rows,
        SUM(CASE WHEN headline_body_cosine_sim IS NULL THEN 1 ELSE 0 END)    AS null_headline_sim,
        SUM(CASE WHEN title_body_cosine_sim IS NULL THEN 1 ELSE 0 END)       AS null_title_sim,
        SUM(CASE WHEN body_embedding_norm IS NULL THEN 1 ELSE 0 END)         AS null_body_norm,
        ROUND(MIN(headline_body_cosine_sim),  3)  AS min_headline_sim,
        ROUND(MAX(headline_body_cosine_sim),  3)  AS max_headline_sim,
        ROUND(AVG(headline_body_cosine_sim),  3)  AS avg_headline_sim,
        ROUND(AVG(title_body_cosine_sim),     3)  AS avg_title_sim,
        ROUND(AVG(body_embedding_norm),       3)  AS avg_body_norm
    FROM gold.review_embedding_features
""").df()

In [ ]:
# Feature distributions split by label.
con.execute("""
    SELECT
        label,
        COUNT(*)                                     AS reviews,
        ROUND(AVG(headline_body_cosine_sim), 4)      AS avg_headline_body_sim,
        ROUND(AVG(title_body_cosine_sim),    4)      AS avg_title_body_sim,
        ROUND(AVG(body_embedding_norm),      3)      AS avg_body_norm
    FROM gold.review_embedding_features
    WHERE label IS NOT NULL
    GROUP BY label
    ORDER BY label DESC
""").df()

In [ ]:
gold_dir = Path("../../data/gold")
gold_dir.mkdir(parents=True, exist_ok=True)

today = date.today().isoformat()

emb_df = pd.DataFrame({
    "_index":             silver_df["_index"].values,
    "body_embedding":     list(body_emb),
    "headline_embedding": list(headline_emb),
    "title_embedding":    list(title_emb),
})

emb_output = gold_dir / f"embeddings_load_date={today}.parquet"
emb_df.to_parquet(emb_output, index=False)
print(f"Raw embeddings saved: {emb_output.resolve()}")
print(f"Shape:  {emb_df.shape[0]:,} rows x {body_emb.shape[1]} dims per field")

In [ ]:
sim_output = gold_dir / f"embedding_features_load_date={today}.parquet"
con.execute(f"""
    COPY (SELECT * FROM gold.review_embedding_features)
    TO '{sim_output.as_posix()}' (FORMAT PARQUET)
""")
print(f"Similarity features saved: {sim_output.resolve()}")

In [ ]:
con.close()